# Broadcast join, AQE e cache

## Task 3: remover a configuração herdada, reaproveitar resultado com cache e registrar o antes e o depois

In [ ]:
# a sessão Spark deste notebook: já vem pronta (é o código do vídeo 1.4, com outro appName)
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-otimizacao")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 4.1 (feed, partidas, configuração herdada e relatório).

In [ ]:
# vídeo 4.1: o feed e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.1: a configuração herdada do job
# configuração herdada do job da operação
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# vídeo 4.1: o relatório da operação
def relatorio_da_operacao():
    """Quantos lances de cada tipo aconteceram em cada Copa."""
    return (
        eventos.join(partidas, on="match_id")
        .groupBy("ano", "event_type")
        .count()
    )

A linha de base desta sessão (`-o` guarda o resultado numa variável):

## Broadcast join: o join sem shuffle no lado grande

Uma **cópia inteira** da tabela pequena vai para cada executor, e o lado grande não é embaralhado.

- o join agora é um **`BroadcastHashJoin`**;
- o caminho do feed **não tem `Exchange`**;
- o **`BroadcastExchange`** é a cópia da tabela pequena, não um shuffle.

O Spark faz o broadcast sozinho abaixo de `spark.sql.autoBroadcastJoinThreshold` (padrão: **10 MB**). O `unset` volta
a configuração ao padrão:

## AQE: o Spark ajustando o plano durante a execução

O **AQE** refaz partes do plano durante a execução, com o tamanho real dos dados:

1. junta partições pequenas depois de um shuffle;
2. troca a estratégia de join;
3. divide partições grandes demais em joins desequilibrados (vídeo 4.4).

Depois de uma ação, o `explain()` mostra o plano final do AQE:

`isFinalPlan=true`: o plano que rodou. **`AQEShuffleRead coalesced`**: o AQE juntando as partições pequenas.

## Registrando o antes e o depois

## Cache: pagar uma vez, reusar várias

**Cada ação executa o plano de novo** (aula 2). O `cache()` guarda o resultado na primeira ação, e as próximas leem de
lá.

1. `cache()` é preguiçoso: nada é guardado até a primeira ação.
2. Ocupa memória: só vale com **reuso**.
3. Libere com `unpersist()`.

A primeira ação depois do `cache()` é a mais lenta (faz o trabalho e guarda); as seguintes leem do cache.

Aba **Storage** da Spark UI: o que está em cache. Depois de usar, libere:

## Resultado da task 3

Configuração herdada removida (broadcast e AQE), medições em `docs/otimizacao.md` e o cache para reuso. Próxima task:
um job que trava numa task só.

## Para consultar depois

- **`persist()`** é o `cache()` com escolha de onde guardar, por exemplo `persist(StorageLevel.MEMORY_ONLY)`, com
  `from pyspark import StorageLevel`.
- **`F.broadcast()`** é útil quando o Spark não consegue estimar o tamanho de uma tabela pequena (vídeo 4.4).
- **Configurações do AQE:** `spark.sql.adaptive.coalescePartitions.enabled` (juntar partições) e
  `spark.sql.adaptive.skewJoin.enabled` (dividir partições grandes) vêm ligadas por padrão.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()